# ASTRA AION-ARC Ω — Reconnexion Kaggle + vérification GPU gratuite

**Mission (4 novembre 2026)** : réutiliser le compte Kaggle `drstarck` sans mot de passe dans ChatGPT, puis vérifier **sur Kaggle** qu'un notebook privé peut réellement démarrer avec un GPU disponible dans le quota gratuit.

**Ce notebook réutilise les briques existantes.** Il essaie d'abord le secret privé Colab `KAGGLE_API_TOKEN` (si le titulaire l'avait déjà enregistré). Sinon, il présente le flux OAuth officiel de Kaggle : **l'approbation du titulaire dans son navigateur est nécessaire**, sans code transmis dans le chat. L'autorisation et ses jetons ne sont jamais publiés dans GitHub.

**Important : le concours ARC-AGI-3 a fermé ses soumissions le 2 novembre 2026 à 23:59 UTC.** Le présent notebook ne soumet rien au concours et n'exécute pas le modèle AION 27B. Il ne fait qu'auditer l'accès Kaggle existant et envoyer **une seule vérification GPU privée** (NvidiaTeslaT4, internet désactivé, sans sources de compétition ni données payantes). Cette vérification consomme éventuellement une petite part du quota GPU Kaggle gratuit ; elle n'achète rien.

**Prérequis :** compte Kaggle déjà détenu par le titulaire et session Google/Colab autorisée. L'accès GPU est soumis aux conditions, à la vérification du compte et au quota Kaggle. Un échec de GPU n'est pas une preuve que l'authentification a échoué.

**Preuves attendues** : `ASTRA_PRIVATE_KAGGLE_AUTH: PASS`, une version de notebook Kaggle `drstarck/astra-arc3-gpu-authorization-20261104` aboutie, et `astra_arc3_gpu_receipt.json` contenant `gpu_available=true` et le modèle de GPU. La simple création d'un notebook ou un CI vert ne vaut pas preuve GPU.

**Mode d'emploi :** « Exécuter tout » dans Colab. Si un écran Kaggle OAuth apparaît, ouvrir le lien d'autorisation dans son propre navigateur, approuver et saisir le code **dans le champ privé Colab uniquement**. Aucune autre opération n'est nécessaire si les permissions GPU et le quota sont disponibles.


In [ ]:
import os, sys, subprocess, pathlib, shutil
ROOT = pathlib.Path("/content/AION-ARC")
BRANCH = "astra/arc3-emergency-20261031"
REMOTE = "https://github.com/Dr-starck66/AION-ARC.git"
def astra_call(argv, *, cwd=None, check=True):
    p = subprocess.run(argv, cwd=cwd, check=check)
    return p.returncode
print("ASTRA_COLAB_SETUP_START", flush=True)
astra_call([sys.executable, "-m", "pip", "install", "-q", "kaggle>=2.2.5", "six"])
if not ROOT.exists():
    astra_call(["git", "clone", "--quiet", "--depth", "1", "--single-branch", "--branch", BRANCH, REMOTE, str(ROOT)])
else:
    astra_call(["git", "fetch", "--quiet", "--depth", "1", "origin", BRANCH], cwd=ROOT)
    astra_call(["git", "checkout", "--quiet", "-B", BRANCH, "FETCH_HEAD"], cwd=ROOT)
assert (ROOT/"kaggle/v29-milestone2/submission.ipynb").is_file()
assert (ROOT/"scripts/arc3_kaggle_final_run.py").is_file()
print("ASTRA_COLAB_SETUP_PASS", flush=True)


In [ ]:
# Reuse the account owner's saved Colab secret when it exists. It is never printed.
import os, subprocess, shutil
try:
    from google.colab import userdata
    saved = userdata.get("KAGGLE_API_TOKEN")
    if saved:
        os.environ["KAGGLE_API_TOKEN"] = saved
        print("ASTRA_AUTH_SOURCE: PRIVATE_COLAB_SECRET", flush=True)
    del saved
except Exception:
    print("ASTRA_AUTH_SOURCE: NO_ACCESSIBLE_COLAB_SECRET", flush=True)
if not shutil.which("kaggle"):
    raise RuntimeError("Kaggle CLI unavailable")
probe = subprocess.run(["kaggle", "kernels", "status", "drstarck/notebook2700e63354"],
                       text=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if probe.returncode != 0:
    print("ASTRA_OAUTH_REQUIRED: official Kaggle authorization link follows.", flush=True)
    # OAuth requires one browser approval and a verification code. Keep the
    # code in the private Colab session, never in ChatGPT or a public log.
    import getpass
    from collections import deque
    # DO NOT use --force; preserve preexisting credentials when available.
    oauth = subprocess.Popen(["kaggle", "auth", "login", "--no-launch-browser"],
                             stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1,
                             env={**os.environ, "PYTHONUNBUFFERED": "1"})
    tail = deque(maxlen=190)
    needle = "Once finished, enter the verification code provided in your browser:"
    while True:
        char = oauth.stdout.read(1)
        if not char:
            raise RuntimeError("Kaggle OAuth did not produce a code prompt; check CLI output")
        print(char, end="", flush=True)
        tail.append(char)
        if needle in "".join(tail):
            break
    # Masked input; no credentials are displayed or committed to GitHub.
    code = getpass.getpass("Kaggle verification code (private Colab prompt): ").strip()
    if not code:
        oauth.kill()
        raise RuntimeError("OAuth authorization not completed; no submission")
    oauth.stdin.write(code + "\n")
    oauth.stdin.flush()
    code = None
    try:
        oauth.wait(timeout=90)
    except subprocess.TimeoutExpired:
        oauth.kill()
        raise RuntimeError("Kaggle OAuth exchange timed out")
    if oauth.returncode != 0:
        raise RuntimeError("Kaggle OAuth rejected the authorization; no submission")
    probe = subprocess.run(["kaggle", "kernels", "status", "drstarck/notebook2700e63354"],
                           text=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if probe.returncode != 0:
    raise RuntimeError("Kaggle auth not verified for owner drstarck; no upload or submission")
print("ASTRA_PRIVATE_KAGGLE_AUTH: PASS (drstarck historic notebook)", flush=True)


In [ ]:
# Read-only: prove Kaggle CLI can access the user's private notebook and
# read the *current official* submission history. No GPU run, push or submit.
import sys, subprocess, json
assert (ROOT / "scripts/arc3_kaggle_final_run.py").is_file()
p = subprocess.run([sys.executable, "scripts/arc3_kaggle_final_run.py", "--check-auth"],
                   cwd=ROOT)
if p.returncode != 0:
    raise RuntimeError("Kaggle account connection not verified. No upload or submission attempted.")
receipt = ROOT / "reports/arc3-final-20261101/receipt.json"
if receipt.is_file():
    data = json.loads(receipt.read_text())
    summary = {k: data[k] for k in
               ("private_kernel_auth", "auth_probe", "submissions_read",
                "recent_submissions", "at_utc") if k in data}
    print("ASTRA_KAGGLE_AUTH_RECEIPT", json.dumps(summary, ensure_ascii=False))
print("ASTRA_COLAB_AUTH_AUDIT_PASS; no changes made to Kaggle", flush=True)


In [ ]:
# ASTRA Ω — ONE private Kaggle GPU test (no competition submission, no paid GPU).
# This cell runs only if the account owner has authorized the preceding Kaggle check.
import json, os, pathlib, subprocess, textwrap, time
from datetime import datetime, timezone

gpu_kernel = "drstarck/astra-arc3-gpu-authorization-20261104"
gpu_folder = pathlib.Path("/content/astra-arc3-gpu-authorization-20261104")
gpu_folder.mkdir(parents=True, exist_ok=True)

# The private historical notebook is a stable account-ownership probe.
owner_probe = subprocess.run(
    ["kaggle", "kernels", "status", "drstarck/notebook2700e63354"],
    capture_output=True, text=True, timeout=60,
)
if owner_probe.returncode != 0:
    raise RuntimeError("FAIL_AUTH: Kaggle CLI cannot access the owner's private historical notebook. No GPU run attempted.")

gpu_source = textwrap.dedent("""
    import json
    from datetime import datetime, timezone
    from pathlib import Path
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("KAGGLE_GPU_UNAVAILABLE: torch.cuda.is_available() returned false")
    evidence = {
        "status": "PASS",
        "scope": "kaggle_private_gpu_smoke_test_not_aion_solver",
        "gpu_available": True,
        "gpu_count": torch.cuda.device_count(),
        "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        "pytorch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "aion_v25_v26_score": "UNVERIFIED",
        "official_competition_score": "UNVERIFIED",
    }
    Path("/kaggle/working/astra_arc3_gpu_receipt.json").write_text(
        json.dumps(evidence, indent=2) + "\\n", encoding="utf-8"
    )
    print("ASTRA_KAGGLE_GPU_SMOKE_PASS", json.dumps(evidence, ensure_ascii=False))
""")
(gpu_folder / "probe.py").write_text(gpu_source, encoding="utf-8")
gpu_metadata = {
    "id": gpu_kernel,
    "title": "astra-arc3-gpu-authorization-20261104",
    "code_file": "probe.py",
    "language": "python",
    "kernel_type": "script",
    "is_private": True,
    "enable_gpu": True,
    "enable_internet": False,
    "machine_shape": "NvidiaTeslaT4",
    "dataset_sources": [],
    "competition_sources": [],
    "kernel_sources": [],
    "model_sources": [],
}
(gpu_folder / "kernel-metadata.json").write_text(
    json.dumps(gpu_metadata, indent=2) + "\n", encoding="utf-8"
)
assert gpu_metadata["enable_gpu"] and not gpu_metadata["enable_internet"]
assert gpu_metadata["is_private"] and not gpu_metadata["competition_sources"]
assert "competitions submit" not in gpu_source

print("ASTRA_KAGGLE_AUTH_OWNER_PASS; requesting ONE free private T4 GPU notebook", flush=True)
pushed = subprocess.run(
    ["kaggle", "kernels", "push", "-p", str(gpu_folder),
     "--accelerator", "NvidiaTeslaT4", "--timeout", "600"],
    capture_output=True, text=True, timeout=180,
)
if pushed.returncode != 0:
    raise RuntimeError(
        f"GPU_SUBMISSION_BLOCKED: Kaggle rejected the private GPU kernel push (exit {pushed.returncode}). "
        "Inspect account verification, Kaggle GPU quota and kernel UI; no paid resource requested."
    )
print("ASTRA_KAGGLE_GPU_PUSH_ACCEPTED_NOT_YET_GPU_PASS", gpu_kernel, flush=True)

# A completed kernel is still not GPU proof; retrieve the actual machine receipt.
last_status = ""
for _ in range(30):
    checked = subprocess.run(
        ["kaggle", "kernels", "status", gpu_kernel],
        capture_output=True, text=True, timeout=60,
    )
    last_status = (checked.stdout + "\n" + checked.stderr).lower()
    if checked.returncode == 0 and any(word in last_status for word in ("complete", "succeeded")):
        break
    if any(word in last_status for word in ("error", "failed", "cancelled")):
        raise RuntimeError("GPU_RUN_FAIL: Kaggle reports notebook execution failure; check Kaggle logs.")
    time.sleep(20)
else:
    raise RuntimeError("GPU_RUN_UNVERIFIED: Kaggle notebook did not finish during this audit window.")

gpu_output = gpu_folder / "verified_output"
gpu_output.mkdir(exist_ok=True)
got = subprocess.run(
    ["kaggle", "kernels", "output", gpu_kernel, "-p", str(gpu_output)],
    capture_output=True, text=True, timeout=180,
)
if got.returncode != 0:
    raise RuntimeError("GPU_RECEIPT_UNVERIFIED: Kaggle completed but output could not be retrieved.")
receipts = list(gpu_output.rglob("astra_arc3_gpu_receipt.json"))
if len(receipts) != 1:
    raise RuntimeError("GPU_RECEIPT_UNVERIFIED: Exact Kaggle GPU receipt not found.")
gpu_receipt = json.loads(receipts[0].read_text(encoding="utf-8"))
if not (
    gpu_receipt.get("gpu_available") is True
    and isinstance(gpu_receipt.get("gpu_count"), int)
    and gpu_receipt["gpu_count"] >= 1
    and gpu_receipt.get("status") == "PASS"
):
    raise RuntimeError("GPU_RECEIPT_FAIL: Kaggle runtime did not prove an actual GPU.")
print("ASTRA_KAGGLE_GPU_VERIFIED_PASS", json.dumps(gpu_receipt, ensure_ascii=False), flush=True)
